# Explore an annotation run

Reads a prediction JSONL written by `annotate_openrouter.py`, flattens the relations
into tables, checks the evidence quotes against the source text, and writes an Excel
workbook with one sheet per view.

Set `RUN_PATH` in the next cell to the run you want to inspect.

In [1]:
import json
import re
import unicodedata
from collections import Counter
from pathlib import Path

import pandas as pd

# Works whether the notebook is run from src/ or from the repository root.
REPO = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()

RUN_PATH = REPO / "runs" / "predictions_chr_annotated.jsonl"
CORPUS_PATH = REPO / "data" / "corpus_chr.jsonl"
DATA_PATH = REPO / "data" 
#XLSX_PATH = REPO / "runs" / "gide_opus5_relations.xlsx"

CLOSED_FIELDS = [
    "relation_type", "structure",
    "status", "reciprocity", "exclusivity", "adultery",
    "explicitness", "explicitness_in_plot",
    "centrality", "framing", "outcome",
    "confidence",
]
REL_COLUMNS = ["novel_id","orientation", "model", "relation_id", "characters"] + CLOSED_FIELDS + ["n_evidence", "comment"]
EV_COLUMNS = ["novel_id", "relation_id", "evidence_index", "quote", "locator", "description"]

pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 220)

print("run    :", RUN_PATH, "| exists:", RUN_PATH.exists())
print("corpus :", CORPUS_PATH, "| exists:", CORPUS_PATH.exists())

run    : c:\Users\plathoon\Desktop\Aurelien\annotation_literature_relationships-main\runs\predictions_chr_annotated.jsonl | exists: True
corpus : c:\Users\plathoon\Desktop\Aurelien\annotation_literature_relationships-main\data\corpus_chr.jsonl | exists: True


## 1. Load the run and check it succeeded

A failed request still produces a record, with the reason in `runtime_error`,
`parse_error`, or `validation_error`. Check this before reading the tables below.

In [2]:
records = [
    json.loads(line)
    for line in RUN_PATH.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
print(f"{len(records)} record(s)\n")

n_failed = 0
for r in records:
    print(f"{r.get('novel_id')} | {r.get('model')} | ok={r.get('ok')}")
    for key in ("runtime_error", "parse_error", "validation_error"):
        if r.get(key):
            n_failed += 1
            print(f"    {key}: {str(r[key])[:800]}")
    usage = (r.get("raw_response") or {}).get("usage")
    if usage:
        print(f"    usage: {usage}")

if n_failed:
    print("\n>> This run has failures. The tables below will be empty until it is re-run.")

5 record(s)

delarue_anges | deepseek/deepseek-v4-pro | ok=True
    usage: {'prompt_tokens': 62831, 'completion_tokens': 4446, 'total_tokens': 67277, 'cost': 0.08359295, 'is_byok': False, 'prompt_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'audio_tokens': 0, 'video_tokens': 0}, 'cost_details': {'upstream_inference_cost': 0.08359295, 'upstream_inference_prompt_cost': 0.07225565, 'upstream_inference_completions_cost': 0.0113373}, 'completion_tokens_details': {'reasoning_tokens': 0, 'image_tokens': 0, 'audio_tokens': 0}}
delarue_anges | anthropic/claude-sonnet-5 | ok=True
    usage: {'prompt_tokens': 98715, 'completion_tokens': 19341, 'total_tokens': 118056, 'cost': 0.39084, 'is_byok': False, 'prompt_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'audio_tokens': 0, 'video_tokens': 0}, 'cost_details': {'upstream_inference_cost': 0.39084, 'upstream_inference_prompt_cost': 0.19743, 'upstream_inference_completions_cost': 0.19341}, 'completion_tokens_details'

## 2. Relations table

One row per annotated relationship, with the eleven closed variables grouped as in the codebook.

In [3]:
# def relation_rows(records):
#     for r in records:
#         annotation = r.get("annotation") or {}
#         for rel in annotation.get("relations") or []:
#             row = {
#                 "novel_id": r.get("novel_id"),
#                 'orientation': r.get("orientation"),
#                 "model": r.get("model"),
#                 "relation_id": rel.get("relation_id"),
#                 "characters": " | ".join(str(c) for c in rel.get("characters") or []),
#                 "n_evidence": len(rel.get("evidence") or []),
#                 "comment": rel.get("comment"),
#             }
#             row.update({f: rel.get(f) for f in CLOSED_FIELDS})
#             yield row


# relations = pd.DataFrame(list(relation_rows(records)), columns=REL_COLUMNS)
# print(f"{len(relations)} relation(s)")
# relations.to_excel(DATA_PATH / "relations_chr_5models.xlsx", index=False, engine="openpyxl")
relations = pd.read_excel(DATA_PATH / "relations_chr_5models.xlsx", engine="openpyxl")


In [4]:
relations_gold = pd.read_excel(REPO / 'annotations' / "delarue_anges_annotation_relationships_correction.xlsx", engine="openpyxl")
relations_gold

,novel_id,title,author,year,relation_id,characters,relation_type,explicitness,centrality,framing,outcome,evidence,confidence,comment,status,structure,reciprocity,exclusivity,adultery,id_uniforme
0,delarue_mardrus_ange_pervers,L'Ange et les pervers,Lucie Delarue-Mardrus,1930,R1,Marion / Mario de Valdeclare / Miss Marion Hervin / Marion Hervin ...,queer,explicit,main,redemptive_spiritualized,stable_or_open,Laurette répète son attachement à Marion et son admiration pour se...,high,Relation centrale et affective entre Laurette et Marion/Mario. La ...,romantic_relationship,dyadic,reciprocal,primary_relationship,loyal,marion_laurette
1,delarue_mardrus_ange_pervers,L'Ange et les pervers,Lucie Delarue-Mardrus,1930,R2,Laurette Wells | Aimée de Lagres,female_female,explicit,secondary,predatory_or_corrupting,illness_madness_decline,Laurette a fait divorcer Aimée et l'a quittée quatre mois après; l...,high,"Relation lesbienne explicitement passionnelle, possessive et manip...",romantic_relationship,dyadic,asymetrical,unclear,unclear,laurette_aimee
2,delarue_mardrus_ange_pervers,L'Ange et les pervers,Lucie Delarue-Mardrus,1930,R3,Aimée de Lagres | comtesse Talliard,female_female,explicit,secondary,neutral_ambivalent,return_to_norm,Aimée et la comtesse Talliard commencent une relation après que La...,high,Autre relation féminine du réseau d'Aimée. Elle est moins développ...,romantic_relationship,dyadic,asymetrical,exclusive,adultery_one,aimee_talliard
3,delarue_mardrus_ange_pervers,L'Ange et les pervers,Lucie Delarue-Mardrus,1930,R4,Aimée de Lagres | ex-mari de Mme de Lagres,female_male,explicit,secondary,neutral_ambivalent,return_to_norm,"Aimée avait quitté son mari pour Laurette, puis a quitté la comtes...",high,Relation conjugale hétérosexuelle qui fonctionne comme retour à la...,married,dyadic,asymetrical,exclusive,adultery_one,aimee_exmari
4,delarue_mardrus_ange_pervers,L'Ange et les pervers,Lucie Delarue-Mardrus,1930,R5,Aimée de Lagres | père non nommé de l'enfant,female_male,explicit,secondary,moralized_pathologized,separation_erasure,Aimée annonce qu'elle est enceinte de quatre mois et demi; l'enfan...,high,Relation sexuelle presque entièrement reconstruite par ses conséqu...,physical_relationship,dyadic,reciprocal,adultery,none,aimee_pereenfant
5,delarue_mardrus_ange_pervers,L'Ange et les pervers,Lucie Delarue-Mardrus,1930,R6,Laurette Wells | Janine / Natacha / Margaret Watson autres admirat...,female_female,coded,milieu,predatory_or_corrupting,stable_or_open,Janine est installée intimement auprès de Laurette; Laurette compl...,medium,"Milieu féminin autour de Laurette, entre désir, domination mondain...",unclear,hierarchy_group,asymetrical,uncommitted,none,laurette_groupe
6,delarue_mardrus_ange_pervers,L'Ange et les pervers,Lucie Delarue-Mardrus,1930,R7,Laurette Wells | Cecil Hampton / Lord Hampton,female_male,explicit,secondary,neutral_ambivalent,separation_erasure,Lord Hampton veut épouser Laurette depuis des années; Laurette dit...,medium,Relation asymétrique: désir et projet matrimonial du côté de Cecil...,seduction,dyadic,one_sided,uncommitted,none,laurette_hampton
7,delarue_mardrus_ange_pervers,L'Ange et les pervers,Lucie Delarue-Mardrus,1930,R8,Marion / Mario de Valdeclare / Miss Marion Hervin / Marion Hervin ...,queer,explicit,episode,eroticized_exoticized,separation_erasure,"Le petit brun dévore Mario des yeux, dit que le revoir sera son id...",high,Désir unilatéral envers Mario/Marion.,seduction,dyadic,one_sided,uncommitted,none,marion_brun
8,delarue_mardrus_ange_pervers,L'Ange et les pervers,Lucie Delarue-Mardrus,1930,R9,Marion / Mario de Valdeclare / Miss Marion Hervin / Marion Hervin ...,queer,coded,milieu,eroticized_exoticized,stable_or_open,"Chez Julien Midalge, un groupe d'hommes poudrés et maniérés se réu...",medium,Milieu homosexuel/queer masculin codé et collectif où règne une at...,seduction,group_or_milieu,reciprocal,uncommitted,none,marion_midalge
9,delarue_mardrus_ange_pervers,L'Ange et les pervers,Lucie Del

## 3. Matching key: `id_uniforme`

Stage 1 of [metric_formalization.md](../metric_formalization.md) matches gold and predicted
relations by an exact key $\kappa(r)$ built from the participant set (§3.1). The gold file
carries that key as `id_uniforme` (e.g. `laurette_aimee`), built by hand from `characters`;
`relations_chr_5models.xlsx` now carries the same column on the LLM side, so the cells
below run for real.

Two things worth keeping in mind when reading the results:
- `novel_id` differs between the two tables (`delarue_anges` in the run vs.
  `delarue_mardrus_ange_pervers` in the gold file) — handled by `NOVEL_ID_MAP` below.
- the gold file predates the v2 schema and has no `explicitness_in_plot`; the code below
  drops any grid field missing on either side instead of failing on it.
- a couple of `id_uniforme` values look like near-misses of each other rather than of the
  gold key (e.g. a model's `marion_mildage` next to gold's `marion_midalge`) — that is a
  transcription slip in the key, not a modelling error, and it costs that relation a match
  under the exact-key rule.

In [5]:
# Bridges the run's novel_id to the gold file's novel_id. Extend as more gold novels arrive.
NOVEL_ID_MAP = {
    "delarue_anges": "delarue_mardrus_ange_pervers",
}

# The 11 codebook dimensions scored for grid agreement; confidence is metadata, used
# separately in the calibration diagnostics (metric_formalization.md §6.3), not scored here.
GRID_FIELDS = [f for f in CLOSED_FIELDS if f != "confidence"]
missing_in_gold = [f for f in GRID_FIELDS if f not in relations_gold.columns]
if missing_in_gold:
    print(f"Dropping from GRID_FIELDS, absent from the gold file: {missing_in_gold}")
    GRID_FIELDS = [f for f in GRID_FIELDS if f not in missing_in_gold]

FIELD_FAMILY = {
    "relation_type": "participants_and_shape", "structure": "participants_and_shape",
    "status": "nature_and_rules", "reciprocity": "nature_and_rules",
    "exclusivity": "nature_and_rules", "adultery": "nature_and_rules",
    "explicitness": "representation", "explicitness_in_plot": "representation",
    "centrality": "representation", "framing": "representation", "outcome": "representation",
}


def field_weights(mode: str = "family_balanced") -> dict[str, float]:
    """w_k, §4.1: flat (1/K) or family-balanced (1 / 3 * family size)."""
    if mode == "flat":
        return {f: 1 / len(GRID_FIELDS) for f in GRID_FIELDS}
    if mode == "family_balanced":
        family_size = Counter(FIELD_FAMILY[f] for f in GRID_FIELDS)
        return {f: 1 / (3 * family_size[FIELD_FAMILY[f]]) for f in GRID_FIELDS}
    raise ValueError(mode)

Dropping from GRID_FIELDS, absent from the gold file: ['explicitness_in_plot']


## 4. Stage 1 — identification ($F_{1,\mathrm{det}}$)

Exact hash join on `(novel_id, id_uniforme)` (§3.1): no similarity function, no threshold.
`identify_matches` keeps every gold and predicted row through an outer join, per model, so
unmatched rows on either side stay visible as false negatives / false positives.

In [6]:
def identify_matches(gold: pd.DataFrame, pred: pd.DataFrame, id_col: str = "id_uniforme") -> pd.DataFrame:
    """Exact-key match per model (§3.1): outer join on (novel_id, id_col)."""
    gold = gold.rename(columns={"novel_id": "novel_id_gold"})
    matched = []
    for model, pred_model in pred.groupby("model"):
        merged = gold.merge(
            pred_model,
            on=["novel_id_gold", id_col],
            how="outer",
            suffixes=("_gold", "_pred"),
            indicator=True,
        )
        merged["model"] = model
        matched.append(merged)
    return pd.concat(matched, ignore_index=True)


def identification_scores(matches: pd.DataFrame) -> pd.DataFrame:
    """P_det, R_det, F1_det per model (§3.1)."""
    def scores(group: pd.DataFrame) -> pd.Series:
        n_match = (group["_merge"] == "both").sum()
        n_gold = (group["_merge"] != "right_only").sum()
        n_pred = (group["_merge"] != "left_only").sum()
        precision = n_match / n_pred if n_pred else float("nan")
        recall = n_match / n_gold if n_gold else float("nan")
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else float("nan")
        return pd.Series({"n_gold": n_gold, "n_pred": n_pred, "n_match": n_match,
                           "P_det": precision, "R_det": recall, "F1_det": f1})

    return matches.groupby("model").apply(scores).reset_index()


relations["novel_id_gold"] = relations["novel_id"].map(NOVEL_ID_MAP)
matches = identify_matches(relations_gold, relations)
identification_scores(matches)

C:\Users\plathoon\AppData\Local\Temp\ipykernel_10684\1896819464.py:30: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return matches.groupby("model").apply(scores).reset_index()


,model,n_gold,n_pred,n_match,P_det,R_det,F1_det
0,anthropic/claude-opus-5,14.0,14.0,10.0,0.714286,0.714286,0.714286
1,anthropic/claude-sonnet-5,14.0,6.0,6.0,1.000000,0.428571,0.600000
2,deepseek/deepseek-v4-pro,14.0,7.0,6.0,0.857143,0.428571,0.571429
3,openai/gpt-5.6-luna-pro,14.0,5.0,5.0,1.000000,0.357143,0.526316
4,openai/gpt-5.6-terra,14.0,6.0,6.0,1.000000,0.428571,0.600000


In [12]:
GRID_FIELDS

['relation_type',
 'structure',
 'status',
 'reciprocity',
 'exclusivity',
 'adultery',
 'explicitness',
 'centrality',
 'framing',
 'outcome']

## 5. Stage 2 — grid agreement on matched pairs ($\bar A$)

For every matched pair (`_merge == "both"`), average the strict per-field credit
$s_k = \mathbb{1}[\ell_k(g) = \ell_k(p)]$ over `GRID_FIELDS`, weighted by `field_weights`
(§4.2, strict variant only — graded ordinal credit and the `queer` / `unknown_gender`
zero-credit guard are described in the doc as a refinement, not implemented here).

In [15]:
def relation_agreement(row: pd.Series, weights: dict[str, float]) -> float:
    """a(g, p), §4: weighted mean of strict per-field credit on one matched pair."""
    num = sum(weights[f] * (row[f"{f}_gold"] == row[f"{f}_pred"]) for f in GRID_FIELDS)
    return num / sum(weights.values())


weights = field_weights("flat")
matched = matches[matches["_merge"] == "both"].copy()
matched["agreement"] = matched.apply(lambda row: relation_agreement(row, weights), axis=1)
matched[["model", "id_uniforme", "agreement"]]

,model,id_uniforme,agreement
0,anthropic/claude-opus-5,aimee_exmari,0.9
1,anthropic/claude-opus-5,aimee_pereenfant,0.7
2,anthropic/claude-opus-5,aimee_talliard,0.8
3,anthropic/claude-opus-5,ginette_theatre,0.3
5,anthropic/claude-opus-5,laurette_aimee,0.4
6,anthropic/claude-opus-5,laurette_groupe,0.6
7,anthropic/claude-opus-5,laurette_hampton,0.8
9,anthropic/claude-opus-5,marion_brun,0.9
11,anthropic/claude-opus-5,marion_laurette,0.5
16,anthropic/claude-opus-5,pere_mere,0.8


## 6. Composite score ($\mathrm{sF}_1 = \bar A \cdot F_{1,\mathrm{det}}$, Prop. 1)

Read the triple, not `sF1` alone: it is exactly how many of the right relations were found
($F_{1,\mathrm{det}}$) times how well the found ones were described ($\bar A$).

In [16]:
def composite_scores(matches: pd.DataFrame, matched: pd.DataFrame) -> pd.DataFrame:
    """(F1_det, A_bar, sF1, exact-tuple accuracy) per model — Prop. 1."""
    det = identification_scores(matches).set_index("model")
    a_bar = matched.groupby("model")["agreement"].mean().rename("A_bar")
    exact = matched.groupby("model")["agreement"].apply(lambda a: (a == 1.0).mean()).rename("exact_tuple_accuracy")
    out = det.join(a_bar, how="left").join(exact, how="left")
    out["sF1"] = out["A_bar"] * out["F1_det"]
    return out.reset_index()


composite_scores(matches, matched)

C:\Users\plathoon\AppData\Local\Temp\ipykernel_27760\1896819464.py:30: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return matches.groupby("model").apply(scores).reset_index()


,model,n_gold,n_pred,n_match,P_det,R_det,F1_det,A_bar,exact_tuple_accuracy,sF1
0,anthropic/claude-opus-5,14.0,14.0,10.0,0.714286,0.714286,0.714286,0.670000,0.0,0.478571
1,anthropic/claude-sonnet-5,14.0,6.0,6.0,1.000000,0.428571,0.600000,0.666667,0.0,0.400000
2,deepseek/deepseek-v4-pro,14.0,7.0,6.0,0.857143,0.428571,0.571429,0.466667,0.0,0.266667
3,openai/gpt-5.6-luna-pro,14.0,5.0,5.0,1.000000,0.357143,0.526316,0.600000,0.0,0.315789
4,openai/gpt-5.6-terra,14.0,6.0,6.0,1.000000,0.428571,0.600000,0.733333,0.0,0.440000


## 7. `unclear` diagnostics (§4.3): coverage and over-commitment

Per field, on matched pairs: `coverage` = P(prediction commits | gold commits),
`over_commitment` = P(prediction commits | gold says `unclear`), plus accuracy conditional
on both committing — the risk–coverage view that keeps a model that answers `unclear`
everywhere from looking safe.

In [17]:
def unclear_diagnostics(matched: pd.DataFrame) -> pd.DataFrame:
    """cov_k, over_k and accuracy | both commit, per model per field (§4.3)."""
    rows = []
    for model, group in matched.groupby("model"):
        for field in GRID_FIELDS:
            gold_col, pred_col = f"{field}_gold", f"{field}_pred"
            gold_commits = group[gold_col] != "unclear"
            pred_commits = group[pred_col] != "unclear"
            both = gold_commits & pred_commits
            rows.append({
                "model": model,
                "field": field,
                "coverage": pred_commits[gold_commits].mean() if gold_commits.any() else float("nan"),
                "over_commitment": pred_commits[~gold_commits].mean() if (~gold_commits).any() else float("nan"),
                "accuracy_when_both_commit": (
                    (group.loc[both, gold_col] == group.loc[both, pred_col]).mean() if both.any() else float("nan")
                ),
            })
    return pd.DataFrame(rows)


unclear_diagnostics(matched)

,model,field,coverage,over_commitment,accuracy_when_both_commit
0,anthropic/claude-opus-5,relation_type,1.000000,NaN,0.900000
1,anthropic/claude-opus-5,structure,1.000000,NaN,0.900000
2,anthropic/claude-opus-5,status,1.000000,1.0,0.875000
3,anthropic/claude-opus-5,reciprocity,0.800000,NaN,0.250000
4,anthropic/claude-opus-5,exclusivity,1.000000,1.0,0.777778
5,anthropic/claude-opus-5,adultery,1.000000,1.0,0.777778
6,anthropic/claude-opus-5,explicitness,1.000000,NaN,0.700000
7,anthropic/claude-opus-5,centrality,1.000000,NaN,0.900000
8,anthropic/claude-opus-5,framing,1.000000,NaN,0.300000
9,anthropic/claude-opus-5,outcome,1.000000,NaN,0.700000


## 8. Per-field accuracy table

Strict accuracy per model per field — the granularity metric_formalization.md recommends
(§4.2, §7.4) over a single averaged number: expect near-perfect agreement on the factual
fields (`relation_type`, `structure`, `status`) and near-chance on the interpretive ones
(`framing`, `explicitness_in_plot`).

In [11]:
def per_field_accuracy(matched: pd.DataFrame) -> pd.DataFrame:
    """Strict accuracy per model per field, one row per field, one column per model."""
    rows = [
        {
            "model": model,
            "field": field,
            "accuracy": (group[f"{field}_gold"] == group[f"{field}_pred"]).mean(),
            "n": len(group),
        }
        for model, group in matched.groupby("model")
        for field in GRID_FIELDS
    ]
    return pd.DataFrame(rows).pivot(index="field", columns="model", values="accuracy")


per_field_accuracy(matched)

model,anthropic/claude-opus-5,anthropic/claude-sonnet-5,deepseek/deepseek-v4-pro,openai/gpt-5.6-luna-pro,openai/gpt-5.6-terra
field,,,,,
adultery,0.7,0.666667,0.166667,0.4,0.333333
centrality,0.9,1.000000,0.666667,0.8,0.833333
exclusivity,0.7,0.833333,0.000000,0.4,0.500000
explicitness,0.7,0.666667,0.333333,0.8,1.000000
framing,0.3,0.166667,0.166667,0.0,0.500000
outcome,0.7,0.666667,0.333333,0.6,0.833333
reciprocity,0.2,0.166667,0.333333,0.2,0.333333
relation_type,0.9,1.000000,0.833333,1.0,1.000000
status,0.7,0.833333,0.833333,0.8,1.000000
